# Image Cloaking Stage: PGD encoder attack on selfies (FFHQ dataset or user uploads)

This notebook returns **cloaked image** from the input by applying Projected Gradient Descent (PGD). It looks for a perturbation $\delta$ with $\|\delta\|_\infty \le \varepsilon$ that pushes the VAE latent of the image toward the latent of a meaningless target (a flat gray image):

$$\min_{\delta}\; \big\| \mathcal{E}(x+\delta) - \mathcal{E}(x_{\text{gray}}) \big\|_2^2 \quad \text{s.t.}\quad \|\delta\|_\infty \le \varepsilon,\;\; x+\delta \in [0,1]$$

* **Inputs:** (a) images from the Kaggle FFHQ dataset `arnaud58/flickrfaceshq-dataset-ffhq`, downloaded **one file at a time** (no need to pull the whole dataset); (b) any selfies user upload.
* **Encoder attacked:** the VAE of `stable-diffusion-v1-5/stable-diffusion-inpainting`, the same encoder the attacker's inpainting pipeline will use.
* **Mask-agnostic:** the cloak covers the whole image. The defender doesn't know which mask the attacker will use.
* **Invisibility is not tuned in this phase.** The default ε values are large on purpose (8/16/32 ÷ 255) so the cloak clearly works. We'll tune ε for imperceptibility later.

**Outputs** (all under `OUT_DIR`):
```
images/<img_id>/clean.png            512×512 preprocessed original  ← use THIS as the clean input downstream
images/<img_id>/cloak_eps<E>.png     PGD cloak, quantized to uint8 PNG
images/<img_id>/random_eps<E>.png    random ±ε noise with the same budget (control: is it the *adversarial* direction or just noise?)
manifest.csv                         one row per (image, ε): paths + latent losses + PSNR + L∞
config.json                          every setting used for this run
figures/                             sanity-check grids and loss curves
```

**Before running:** `Runtime → Change runtime type → T4 GPU`. If the Kaggle download fails, see the note in section 3.

## 1. Install

In [1]:
!pip install -q -U diffusers accelerate kagglehub pillow-heif

In [2]:
import os, io, re, json, time, math, zlib, random, shutil, contextlib
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
import matplotlib.pyplot as plt

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={device} | torch={torch.__version__} | Colab={IN_COLAB}")
if device == "cpu":
    print("WARNING: no GPU, so PGD will be very slow. Runtime > Change runtime type > T4 GPU.")

device=cuda | torch=2.14.1+cu130 | Colab=True


## 2. Configuration
Everything you might want to change is here. Set `QUICK_TEST = True` for a ~2-minute smoke test (2 images, one ε, 50 iterations).

In [3]:
# ===== Input settings  =====
USE_FFHQ       = True
KAGGLE_DATASET = "arnaud58/flickrfaceshq-dataset-ffhq"
FFHQ_IDS       = ["00004"]        # always included (file names without ".png")
N_FFHQ_RANDOM  = 34               # extra FFHQ images sampled at random → 35 FFHQ images total
FFHQ_ID_RANGE  = (0, 50000)       # random ids drawn from this range
USE_UPLOAD     = False            # open a file picker for selfies (jpg/png/heic, several at once is fine)
SELFIE_PATHS   = []               # list of local selfie paths

# ===== Cloak: PGD encoder attack =====
MODEL_ID    = "stable-diffusion-v1-5/stable-diffusion-inpainting"   # attack the VAE the attacker will use
IMG_SIZE    = 512
EPS_LIST    = [8, 16, 32]   # L∞ budgets, in 1/255 units on the [0,1] pixel scale (PhotoGuard's 0.06 on [-1,1] ≈ 8/255)
N_ITERS     = 200
STEP_FRAC   = 0.125         # step α = STEP_FRAC·ε; α is halved at 50% and again at 75% of the iterations
RANDOM_INIT = True          # start from uniform noise in [-ε, ε]
TARGET      = "gray"        # "gray" = flat 0.5 image, or a path to any target image
BATCH_SIZE  = 4             # T4 handles 4 at 512² with fp16; lowered automatically on out-of-memory
USE_FP16    = True          # autocast fp16 for the encoder (≈4× faster on T4); set False if losses become NaN
SAVE_RANDOM_BASELINE = True # also save random ±ε noise images with the same budget
SEED        = 0
QUICK_TEST  = False

# ===== Output =====
SAVE_TO_DRIVE = True       # True = write to Google Drive so results survive Colab disconnects (and runs can resume)
OUT_DIR = "/content/drive/MyDrive/CRPII/cloak_out" if SAVE_TO_DRIVE else "/content/cloak_out"

if QUICK_TEST:
    FFHQ_IDS, N_FFHQ_RANDOM, EPS_LIST, N_ITERS = FFHQ_IDS[:1], 1, [16], 50

In [4]:
if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
if not IN_COLAB and OUT_DIR.startswith("/content"):
    OUT_DIR = os.path.abspath("cloak_out")

IMG_DIR  = os.path.join(OUT_DIR, "images")
FIG_DIR  = os.path.join(OUT_DIR, "figures")
MANIFEST = os.path.join(OUT_DIR, "manifest.csv")
os.makedirs(IMG_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
print("Writing results to", OUT_DIR)

Mounted at /content/drive
Writing results to /content/drive/MyDrive/CRPII/cloak_out


## 3. Collect input images
Every image is converted the same way: EXIF rotation fixed, center-cropped to a square, resized to 512×512 (LANCZOS) and saved as `clean.png`. Downstream notebooks (breakers, inpainting, evaluation) should load this `clean.png` (not raw file), so the clean image, the cloaked image and the mask stay pixel-aligned.

In [5]:
def to_square(img, size=IMG_SIZE):
    # Ensure selfie image is upright
    img = ImageOps.exif_transpose(img).convert("RGB")
    w, h = img.size
    # crop center square of image
    s = min(w, h)
    left, top = (w - s) // 2, (h - s) // 2
    img = img.crop((left, top, left + s, top + s))
    if s != size:
        img = img.resize((size, size), Image.LANCZOS)
    return img

# img_id -> dict(img_id, source, original, clean_path)
items = {}
def register_image(img_id, source, src_path=None):
    """Catalog raw input image into clean and standardized format
        Args:
            img_id (string): unique identifier for the image
            source (string): source of image
            src_path (string): Drive file path of image

        Returns:
            items (Dict)
    """
    # create dir
    d = os.path.join(IMG_DIR, img_id)
    os.makedirs(d, exist_ok=True)
    # construct path
    clean_path = os.path.join(d, "clean.png")
    # crop, resize and save
    if not os.path.exists(clean_path):
        with Image.open(src_path) as im:
            to_square(im).save(clean_path)
    # register in global dict
    items[img_id] = dict(img_id=img_id, source=source,
                         original=os.path.basename(src_path) if src_path else "",
                         clean_path=clean_path)

### 3a. FFHQ from Kaggle (single-file downloads)
`kagglehub` downloads only the files you ask for. Public datasets often download without logging in. If you get a **401/403 error**:
1. On kaggle.com go to **Settings → API → Create New Token**.
2. In Colab, open the 🔑 **Secrets** panel and add `KAGGLE_USERNAME` and `KAGGLE_KEY` (or `KAGGLE_API_TOKEN`). Turn on notebook access for both.
3. Re-run this cell.

FFHQ images are individually licensed under Creative Commons, and the dataset itself is BY-NC-SA. Keep the ids in `manifest.csv` for attribution.

In [ ]:
def fetch_ffhq(file_id):
    # Fetch dataset from Github
    import kagglehub
    return kagglehub.dataset_download(KAGGLE_DATASET, path=f"{file_id}.png")

def add_ffhq(fid):
    # Add ffhq tag to image
    img_id = f"ffhq_{fid}"
    if os.path.exists(os.path.join(IMG_DIR, img_id, "clean.png")):   # already prepared (resume)
        register_image(img_id, "ffhq")
        items[img_id]["original"] = f"{fid}.png"
        return True
    try:
        register_image(img_id, "ffhq", fetch_ffhq(fid))
        return True
    except Exception as e:
        print(f"  ✗ {fid}.png: {type(e).__name__}: {str(e)[:160]}")
        return False

if USE_FFHQ:
    got = [fid for fid in FFHQ_IDS if add_ffhq(fid)]
    # Assert no FFHQ file has been downloaded
    if FFHQ_IDS and not got:
        print("\nNo FFHQ file could be downloaded. Check the Kaggle credentials (see the note above) "
              "and that the file names exist in the dataset.")
    target_n, rng, tried, fails_in_a_row = len(got) + N_FFHQ_RANDOM, random.Random(SEED), set(got), 0
    while len(got) < target_n and fails_in_a_row < 8:
        fid = f"{rng.randrange(*FFHQ_ID_RANGE):05d}"
        if fid in tried:
            continue
        tried.add(fid)
        if add_ffhq(fid):
            got.append(fid); fails_in_a_row = 0
        else:
            fails_in_a_row += 1
    if len(got) < target_n:
        print(f"Stopped after {fails_in_a_row} failures in a row; try lowering FFHQ_ID_RANGE.")
    print(f"FFHQ images ready: {len(got)} → {sorted(got)[:10]}{' …' if len(got) > 10 else ''}")

100%|██████████| 366k/366k [00:00<00:00, 1.10MB/s]


100%|██████████| 378k/378k [00:00<00:00, 1.20MB/s]


100%|██████████| 349k/349k [00:00<00:00, 1.18MB/s]


100%|██████████| 381k/381k [00:00<00:00, 1.31MB/s]


100%|██████████| 392k/392k [00:00<00:00, 1.21MB/s]


100%|██████████| 417k/417k [00:00<00:00, 1.33MB/s]


... (19 outputs left out)

100%|██████████| 323k/323k [00:00<00:00, 1.08MB/s]


100%|██████████| 386k/386k [00:00<00:00, 1.25MB/s]


100%|██████████| 392k/392k [00:00<00:00, 1.23MB/s]


Using Colab cache for faster access to the 'flickrfaceshq-dataset-ffhq' dataset.
Using Colab cache for faster access to the 'flickrfaceshq-dataset-ffhq' dataset.
Using Colab cache for faster access to the 'flickrfaceshq-dataset-ffhq' dataset.
Using Colab cache for faster access to the 'flickrfaceshq-dataset-ffhq' dataset.
Using Colab cache for faster access to the 'flickrfaceshq-dataset-ffhq' dataset.
Using Colab cache for faster access to the 'flickrfaceshq-dataset-ffhq' dataset.
Using Colab cache for faster access to the 'flickrfaceshq-dataset-ffhq' dataset.
Using Colab cache for faster access to the 'flickrfaceshq-dataset-ffhq' dataset.
FFHQ images ready: 35 → ['00004', '02653', '04832', '06214', '06472', '06599', '14315', '16417', '16968', '18470'] …


### 3b. Upload your own selfies

In [ ]:
# if USE_UPLOAD:
#     paths = list(SELFIE_PATHS)
#     if IN_COLAB:
#         from google.colab import files
#         print("Pick one or more selfie files (jpg / png / heic). Click Cancel to skip.")
#         uploaded = files.upload()
#         os.makedirs("/content/selfies_raw", exist_ok=True)
#         for name, data in uploaded.items():
#             p = os.path.join("/content/selfies_raw", name)
#             with open(p, "wb") as f:
#                 f.write(data)
#             paths.append(p)
#     for p in paths:
#         stem = re.sub(r"[^A-Za-z0-9_-]+", "_", os.path.splitext(os.path.basename(p))[0]).strip("_")
#         try:
#             register_image(f"selfie_{stem}", "selfie", p)
#             print("  ✓", p)
#         except Exception as e:
#             print(f"  ✗ {p}: {e}")

# print(f"\nTotal images: {len(items)} "
#       f"({sum(v['source']=='ffhq' for v in items.values())} FFHQ, "
#       f"{sum(v['source']=='selfie' for v in items.values())} selfies)")
# assert items, "No input images: enable FFHQ and/or upload a selfie."

In [ ]:
# def show_grid(paths, titles, ncol=6, size=2.2, save=None):
#     n = len(paths)
#     nrow = math.ceil(n / ncol)
#     fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * size, nrow * size), squeeze=False)
#     for ax in axes.flat:
#         ax.axis("off")
#     for ax, p, t in zip(axes.flat, paths, titles):
#         ax.imshow(Image.open(p)); ax.set_title(t, fontsize=8)
#     plt.tight_layout()
#     if save:
#         fig.savefig(save, dpi=110, bbox_inches="tight")
#     plt.show()

# preview = list(items.values())[:12]
# show_grid([v["clean_path"] for v in preview], [v["img_id"] for v in preview])

## 4. Load the VAE we attack
We only need the VAE (≈80M parameters), not the full diffusion pipeline. The weights stay in fp32; the forward/backward pass runs under fp16 autocast when `USE_FP16=True`.

In [ ]:
from diffusers import AutoencoderKL

vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae").to(device).eval()
vae.requires_grad_(False)
AMP = (device == "cuda" and USE_FP16)
print("VAE loaded:", MODEL_ID, "| fp16 autocast:", AMP)

Disabling torchao because diffusers requires `torchao>=0.15.0` but found version 0.10.0. Upgrade it with `pip install -U 'torchao>=0.15.0'`.
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

An error occurred while trying to fetch stable-diffusion-v1-5/stable-diffusion-inpainting: stable-diffusion-v1-5/stable-diffusion-inpainting does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


vae/diffusion_pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  335MB            

vae/diffusion_pytorch_model.bin: downloading bytes:           |  0.00B            

VAE loaded: stable-diffusion-v1-5/stable-diffusion-inpainting | fp16 autocast: True


## 5. PGD encoder attack
* Pixels live in **[0,1]**; the VAE gets `2x−1`. ε is measured on the [0,1] scale (ε = 8 means 8/255).
* The loss uses the **mean** of the encoder's latent distribution, so it is deterministic. For the backward pass we use the per-image *sum* of squared errors: PGD only uses the gradient's sign, and the sum keeps fp16 gradients from underflowing.
* Each image gets its own gradient (losses are summed across the batch, and images don't interact), so batching doesn't change the result.
* We keep the **best iterate** (lowest latent loss) for each image, then **quantize to uint8**: a real upload is 8-bit, so every number reported below is measured *after* quantization.
* Random init and the random-noise baseline are seeded from `(img_id, ε, SEED)`, so a re-run starts from the same noise whatever the batch composition. The random-noise images are bit-for-bit reproducible. A cloak can differ in a few pixels between runs because GPU floating-point arithmetic isn't fully deterministic, so treat the saved PNGs as the reference copy.

In [ ]:
def amp_ctx():
    return torch.autocast("cuda", dtype=torch.float16) if AMP else contextlib.nullcontext()

def encode_mean(x01):
    with amp_ctx():
        z = vae.encode(x01 * 2 - 1).latent_dist.mean
    return z.float()

def latent_mse(z, z_tgt):
    # per-image mean squared error in latent space
    return ((z - z_tgt) ** 2).flatten(1).mean(1)

def load_batch(paths):
    arr = np.stack([np.asarray(Image.open(p).convert("RGB"), dtype=np.float32) / 255.0 for p in paths])
    return torch.from_numpy(arr).permute(0, 3, 1, 2).contiguous().to(device)

def to_pil(x):
    # # x: (3,H,W) in [0,1]
    return Image.fromarray((x.clamp(0, 1).permute(1, 2, 0).cpu().numpy() * 255).round().astype(np.uint8))

def quantize(x):
    return (x.clamp(0, 1) * 255).round() / 255

def seeded_uniform(img_id, eps_int, salt, shape):
    seed = (zlib.crc32(f"{img_id}|{eps_int}|{salt}|{SEED}".encode())) % (2**31)
    g = torch.Generator().manual_seed(seed)
    return (torch.rand(shape, generator=g) * 2 - 1)

def psnr(a, b):
    # # per-image PSNR in dB, a/b in [0,1]
    mse = ((a - b) ** 2).flatten(1).mean(1).clamp_min(1e-12)
    return 10 * torch.log10(1.0 / mse)

@torch.no_grad()
def build_target_latent(target):
    if target == "gray":
        t = torch.full((1, 3, IMG_SIZE, IMG_SIZE), 0.5, device=device)
    else:
        with Image.open(target) as im:
            arr = np.asarray(to_square(im), dtype=np.float32) / 255.0
        t = torch.from_numpy(arr).permute(2, 0, 1)[None].to(device)
    return encode_mean(t)

def pgd_encoder_attack(x, z_tgt, eps, n_iters, step_frac, init=None):
    # x: (B,3,H,W) in [0,1]; eps on [0,1] scale. Returns best δ (B,3,H,W) and loss history (iters, B).
    alpha0 = step_frac * eps
    delta = torch.zeros_like(x) if init is None else init.to(x.device) * eps
    delta = (x + delta).clamp(0, 1) - x
    best_delta = delta.clone()
    best_loss = torch.full((x.shape[0],), float("inf"), device=x.device)
    history = []
    for it in range(n_iters + 1):
        last = (it == n_iters)
        delta.requires_grad_(not last)
        with torch.set_grad_enabled(not last):
            z = encode_mean(x + delta)
            sq = ((z - z_tgt) ** 2).flatten(1)
            loss = sq.mean(1)
        if torch.isnan(loss).any():
            raise RuntimeError("NaN latent loss: set USE_FP16 = False and re-run.")
        with torch.no_grad():
            lv = loss.detach()
            better = lv < best_loss
            best_loss = torch.where(better, lv, best_loss)
            best_delta[better] = delta.detach()[better]
            history.append(lv.cpu())
        if last:
            break
        grad, = torch.autograd.grad(sq.sum(), delta)
        alpha = alpha0 * (0.5 if it >= n_iters // 2 else 1.0) * (0.5 if it >= 3 * n_iters // 4 else 1.0)
        with torch.no_grad():
            delta = delta.detach() - alpha * grad.sign()
            delta = delta.clamp(-eps, eps)
            delta = (x + delta).clamp(0, 1) - x
    return best_delta, torch.stack(history)

## 6. Run
Safe to re-run: any (image, ε) that already has a PNG and a row in `manifest.csv` is skipped. With `SAVE_TO_DRIVE=True` you can pick up where you left off after a disconnect.

Rough timing on a T4 with fp16 is on the order of a minute per batch of 4 at 200 iterations. The cell prints the measured speed and an ETA after the first batch.

In [ ]:
config = {k: globals()[k] for k in [
    "MODEL_ID", "IMG_SIZE", "EPS_LIST", "N_ITERS", "STEP_FRAC", "RANDOM_INIT", "TARGET",
    "USE_FP16", "SAVE_RANDOM_BASELINE", "SEED", "KAGGLE_DATASET", "FFHQ_IDS", "N_FFHQ_RANDOM", "FFHQ_ID_RANGE"]}

config["attack"] = "PGD encoder attack (PhotoGuard-style), mask-agnostic, L_inf, best iterate, uint8 quantized"
with open(os.path.join(OUT_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

z_tgt = build_target_latent(TARGET)

rows = pd.read_csv(MANIFEST).to_dict("records") if os.path.exists(MANIFEST) else []
done = {(r["img_id"], int(r["eps"])) for r in rows
        if os.path.exists(os.path.join(OUT_DIR, r["cloak_path"]))}
loss_histories = {}            # eps -> list of (iters+1, B) tensors (this session only)

def rel(p):
    return os.path.relpath(p, OUT_DIR)

todo_all = [(eps_int, it) for eps_int in EPS_LIST for it in items.values() if (it["img_id"], eps_int) not in done]
print(f"{len(done)} (image, ε) pairs already done, {len(todo_all)} to do.")

t_start, n_done_now, bs = time.time(), 0, BATCH_SIZE
for eps_int in EPS_LIST:
    eps = eps_int / 255.0
    todo = [it for e, it in todo_all if e == eps_int]
    i = 0
    while i < len(todo):
        batch = todo[i:i + bs]
        try:
            t0 = time.time()
            x = load_batch([b["clean_path"] for b in batch])
            init = (torch.stack([seeded_uniform(b["img_id"], eps_int, "init", x.shape[1:]) for b in batch])
                    if RANDOM_INIT else None)
            delta, hist = pgd_encoder_attack(x, z_tgt, eps, N_ITERS, STEP_FRAC, init)
            x_adv = quantize(x + delta)
            with torch.no_grad():
                L_clean = latent_mse(encode_mean(x), z_tgt)
                L_adv = latent_mse(encode_mean(x_adv), z_tgt)
                if SAVE_RANDOM_BASELINE:
                    signs = torch.stack([seeded_uniform(b["img_id"], eps_int, "rand", x.shape[1:]).sign()
                                         for b in batch]).to(device)
                    x_rnd = quantize(x + eps * signs)
                    L_rnd = latent_mse(encode_mean(x_rnd), z_tgt)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if bs == 1:
                raise
            bs = max(1, bs // 2)
            print(f"Out of GPU memory → batch size lowered to {bs}")
            continue
        dt = time.time() - t0

        for j, b in enumerate(batch):
            d = os.path.dirname(b["clean_path"])
            cloak_path = os.path.join(d, f"cloak_eps{eps_int}.png")
            to_pil(x_adv[j]).save(cloak_path)
            row = dict(
                img_id=b["img_id"], source=b["source"], original=b["original"], eps=eps_int,
                n_iters=N_ITERS, step_frac=STEP_FRAC, target=TARGET,
                clean_path=rel(b["clean_path"]), cloak_path=rel(cloak_path),
                latent_mse_clean=L_clean[j].item(), latent_mse_cloak=L_adv[j].item(),
                loss_ratio_cloak=(L_adv[j] / L_clean[j]).item(),
                linf_255=round(((x_adv[j] - x[j]).abs().max() * 255).item()),
                psnr_cloak=psnr(x_adv[j:j+1], x[j:j+1]).item(),
                seconds_per_image=dt / len(batch),
            )
            if SAVE_RANDOM_BASELINE:
                rnd_path = os.path.join(d, f"random_eps{eps_int}.png")
                to_pil(x_rnd[j]).save(rnd_path)
                row.update(random_path=rel(rnd_path), latent_mse_random=L_rnd[j].item(),
                           loss_ratio_random=(L_rnd[j] / L_clean[j]).item(),
                           psnr_random=psnr(x_rnd[j:j+1], x[j:j+1]).item())
            rows = [r for r in rows if not (r["img_id"] == b["img_id"] and int(r["eps"]) == eps_int)] + [row]
        pd.DataFrame(rows).to_csv(MANIFEST, index=False)       # save after every batch
        loss_histories.setdefault(eps_int, []).append(hist)

        i += len(batch)
        n_done_now += len(batch)
        rate = (time.time() - t_start) / n_done_now
        eta = rate * (len(todo_all) - n_done_now)
        print(f"ε={eps_int:>2}/255 | {i}/{len(todo)} | batch {dt:5.1f}s | "
              f"loss ratio (cloak/clean) {torch.mean(L_adv / L_clean).item():.3f} | ETA {eta/60:5.1f} min")

print(f"\nDone. Manifest: {MANIFEST} ({len(rows)} rows)")

## 7. Did the cloak work? (latent-space check)
`loss_ratio = ‖E(x′) − z_gray‖² / ‖E(x) − z_gray‖²`, measured after uint8 quantization.
* **Cloak ≪ 1:** the encoder now "sees" something close to a gray image. That's what we want.
* **Random ≈ 1:** noise with the same budget barely moves the latent, so the effect comes from the *adversarial direction*, not the perturbation's size.

This is a cheap proxy. The real test is Stage 4 (inpainting) plus ArcFace.

In [ ]:
df = pd.read_csv(MANIFEST)
agg = {"loss_ratio_cloak": "mean", "psnr_cloak": "mean", "linf_255": "max", "img_id": "count"}
if "loss_ratio_random" in df:
    agg.update(loss_ratio_random="mean", psnr_random="mean")
summary = (df.groupby(["source", "eps"]).agg(agg).rename(columns={"img_id": "n_images"}).round(3))
display(summary)

In [ ]:
if loss_histories:
    plt.figure(figsize=(6, 3.5))
    for eps_int, hs in sorted(loss_histories.items()):
        h = torch.cat(hs, dim=1)                     # (iters+1, n_images)
        h = h / h[0:1]                               # normalise each image by its loss at the start
        plt.plot(h.mean(1).numpy(), label=f"ε={eps_int}/255")
    plt.yscale("log"); plt.xlabel("PGD iteration"); plt.ylabel("latent loss / initial (mean)")
    plt.title("PGD convergence"); plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "pgd_convergence.png"), dpi=120)
    plt.show()
else:
    print("No new PGD runs in this session (everything was already done). Nothing to plot.")

## 8. Visual sanity check (the PhotoGuard test)
For a few images: clean | cloaked | |δ| amplified | VAE reconstruction of clean / cloaked / random-noise. If the cloak works, **the reconstruction of the cloaked image should collapse toward a gray, washed-out picture**, while the clean and random-noise versions reconstruct normally.

In [ ]:
SHOW_EPS = EPS_LIST[-1]      # which ε to visualise
N_SHOW   = 4

@torch.no_grad()
def vae_roundtrip(x01):
    with amp_ctx():
        y = vae.decode(vae.encode(x01 * 2 - 1).latent_dist.mean).sample
    return ((y.float() + 1) / 2).clamp(0, 1)

sel = df[df.eps == SHOW_EPS].drop_duplicates("source").head(2)       # one of each source first …
sel = pd.concat([sel, df[(df.eps == SHOW_EPS) & ~df.img_id.isin(sel.img_id)]]).head(N_SHOW)
has_rnd = "random_path" in sel and sel["random_path"].notna().all()
cols = ["clean", f"cloak ε={SHOW_EPS}", "|δ| (amplified)", "VAE recon: clean", "VAE recon: cloak"] + \
       (["VAE recon: random"] if has_rnd else [])
fig, axes = plt.subplots(len(sel), len(cols), figsize=(2.4 * len(cols), 2.5 * len(sel)), squeeze=False)
for r, (_, row) in enumerate(sel.iterrows()):
    paths = [row.clean_path, row.cloak_path] + ([row.random_path] if has_rnd else [])
    xs = load_batch([os.path.join(OUT_DIR, p) for p in paths])
    recon = vae_roundtrip(xs)
    amp = ((xs[1] - xs[0]).abs() * 255 / SHOW_EPS).clamp(0, 1)
    panels = [xs[0], xs[1], amp, recon[0], recon[1]] + ([recon[2]] if has_rnd else [])
    for c, p in enumerate(panels):
        axes[r, c].imshow(to_pil(p)); axes[r, c].axis("off")
        if r == 0:
            axes[r, c].set_title(cols[c], fontsize=9)
    axes[r, 0].text(-8, IMG_SIZE / 2, row.img_id, rotation=90, va="center", ha="right", fontsize=8)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, f"sanity_eps{SHOW_EPS}.png"), dpi=110, bbox_inches="tight")
plt.show()

## 9. Download the results

In [ ]:
if IN_COLAB and not SAVE_TO_DRIVE:
    zip_path = shutil.make_archive("/content/cloak_out", "zip", OUT_DIR)
    from google.colab import files
    files.download(zip_path)
else:
    print("Results are in", OUT_DIR)

## Handing off to the next stages
* **Breakers / inpainting / evaluation:** iterate over `manifest.csv`. For every row, the cloaked branch reads `cloak_path`, the control branch reads `clean_path`, and the noise control reads `random_path`. Paths are relative to `OUT_DIR`.
* Compute the **MediaPipe face mask from `clean.png`** once per image and reuse it for every variant.
* Apply every breaker to **both** the cloak and the clean image.

**Deferred to later phases:** tuning ε for imperceptibility (ε ∈ {4, 8, 16}, plus LPIPS/SSIM), a *mask-aware* ablation (optimise $\mathcal{E}((x+\delta)\odot(1-m))$), and EOT-robust cloaks.